In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from network import generate_graph, generate_activities
from ranking import (
    calculate_ar,
    estimate_cascade_sizes,
    calculate_hybrid,
    select_top_k
)
from evaluation import (
    compare_strategies,
    make_evaluation_trials,
    evaluate_immunization
)

### 1. Set up one experiment

In [ ]:
N = 100
BETA = 0.8
K = 10
M = 300
TRIALS = 1000
ALPHA = 0.5

GRAPH_TYPE = "erdos_renyi"
ACTIVITY_DISTRIBUTION = "mixed"

G = generate_graph(N, GRAPH_TYPE, seed=42)

activities = generate_activities(
    G,
    ACTIVITY_DISTRIBUTION,
    rng=np.random.default_rng(42)
)

print(
    f"Nodes: {G.number_of_nodes()}, "
    f"edges: {G.number_of_edges()}"
)

: 

### 2. Calculate AR, cascade size, and hybrid score

In [ ]:
ar = calculate_ar(
    G, activities, centrality="degree"
)

cascade_sizes = estimate_cascade_sizes(
    G, activities, BETA, simulations=M, seed=43
)

hybrid = calculate_hybrid(
    ar, cascade_sizes, ALPHA
)

rankings = pd.DataFrame({
    "node": list(G.nodes()),
    "activity": [activities[v] for v in G],
    "AR": [ar[v] for v in G],
    "cascade_size": [cascade_sizes[v] for v in G],
    "hybrid": [hybrid[v] for v in G]
})

display(
    rankings.sort_values(
        "hybrid", ascending=False
    ).head(15)
)

### 3. Evaluate the three immunization strategies (AR, Cascade, AR + Cascade Hybrid)

In [ ]:
strategies = {
    "AR": select_top_k(ar, K),
    "Cascade": select_top_k(cascade_sizes, K),
    "Hybrid": select_top_k(hybrid, K)
}

results = compare_strategies(
    G,
    activities,
    BETA,
    strategies,
    trials=TRIALS,
    seed=44
)

display(results)

### 4. Experiment with different alpha

In [ ]:
alphas = np.linspace(0, 1, 11)

sources, random_seeds = make_evaluation_trials(
    G, trials=TRIALS, seed=44
)

alpha_results = []

for alpha in alphas:
    scores = calculate_hybrid(
        ar, cascade_sizes, float(alpha)
    )

    selected = select_top_k(scores, K)

    fraction = evaluate_immunization(
        G,
        activities,
        BETA,
        selected,
        sources,
        random_seeds
    )

    alpha_results.append({
        "alpha": float(alpha),
        "infected_fraction": fraction
    })

alpha_df = pd.DataFrame(alpha_results)

display(alpha_df)

plt.plot(
    alpha_df["alpha"],
    alpha_df["infected_fraction"],
    marker="o"
)

plt.xlabel("Alpha (weight on AR)")
plt.ylabel("Mean infected fraction")
plt.title("Effect of hybrid weighting")
plt.grid(alpha=0.3)
plt.show()

### 5. Compare activity distributions and graph topologies

In [ ]:
records = []

for graph_type in [
    "erdos_renyi",
    "barabasi_albert",
    "watts_strogatz"
]:
    for distribution in [
        "exponential",
        "power_law",
        "negative_binomial",
        "mixed"
    ]:
        for replicate in range(3):
            # Increase to 10–30 for the final report
            seed = 1000 + replicate

            graph = generate_graph(
                N, graph_type, seed=seed
            )

            activity = generate_activities(
                graph,
                distribution,
                rng=np.random.default_rng(seed + 10)
            )

            ar_scores = calculate_ar(
                graph, activity
            )

            cascade = estimate_cascade_sizes(
                graph,
                activity,
                BETA,
                simulations=M,
                seed=seed + 20
            )

            hybrid_scores = calculate_hybrid(
                ar_scores,
                cascade,
                ALPHA
            )

            selected = {
                "AR": select_top_k(ar_scores, K),
                "Cascade": select_top_k(cascade, K),
                "Hybrid": select_top_k(hybrid_scores, K)
            }

            comparison = compare_strategies(
                graph,
                activity,
                BETA,
                selected,
                trials=TRIALS,
                seed=seed + 30
            )

            for row in comparison.to_dict("records"):
                records.append({
                    "graph": graph_type,
                    "distribution": distribution,
                    "replicate": replicate,
                    **row
                })

full_results = pd.DataFrame(records)

summary = full_results.groupby(
    ["graph", "distribution", "strategy"],
    as_index=False
)["mean_infected_fraction"].agg(["mean", "std"])

display(summary)

full_results.to_csv("results.csv", index=False)